# Rip2Alpha Trading Strategy - Consolidated with Multi-Stochastic Breakout

## Complete Strategy Implementation

This notebook contains:
- Original Rip2Alpha strategy (TSBuy/TSSell signals)
- Multi-Stochastic Breakout logic (4 D-lines: 9, 14, 40, 60)
- Comprehensive backtesting framework
- Machine Learning optimization (Take Profit & Indicator Values)
- Advanced parameter optimization
- Performance analysis and visualization

**Simply mount your Google Drive, upload your CSV data, and run all cells!**

## 1. Environment Setup & Library Installation

In [ ]:
# Install required packages for Google Colab
!pip install -q --upgrade pandas numpy matplotlib seaborn ta-lib
!pip install -q pandas-ta scikit-learn scikit-optimize
!pip install -q backtesting

print("✅ All packages installed successfully!")

In [ ]:
# Import all necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from typing import Dict, List, Tuple, Any, Optional
import warnings
warnings.filterwarnings('ignore')

# Technical Analysis Libraries
import talib as ta
import pandas_ta as pta

# Backtesting
from backtesting import Backtest, Strategy
from backtesting.lib import crossover

# Machine Learning & Optimization
from sklearn.model_selection import ParameterGrid
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

try:
    from skopt import gp_minimize
    from skopt.space import Real, Integer
    from skopt.utils import use_named_args
    SKOPT_AVAILABLE = True
    print("✅ Bayesian optimization available")
except ImportError:
    SKOPT_AVAILABLE = False
    print("⚠️ Bayesian optimization not available, using grid search")

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
sns.set_style('darkgrid')
plt.rcParams['figure.figsize'] = (16, 8)

print("\n✅ All imports successful!")

## 2. Mount Google Drive & Load Data

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

print("✅ Google Drive mounted successfully!")

In [ ]:
# Load your data from Google Drive
# MODIFY THIS PATH to point to your CSV file
DATA_PATH = '/content/drive/MyDrive/trading_data/your_data.csv'

def load_and_prepare_data(file_path: str) -> pd.DataFrame:
    """
    Load CSV data and prepare it for backtesting.
    Expects columns: timestamp/date, open, high, low, close, volume
    """
    df = pd.read_csv(file_path)
    
    # Standardize column names
    df.columns = df.columns.str.lower()
    
    # Convert timestamp to datetime if needed
    if 'timestamp' in df.columns:
        df['timestamp'] = pd.to_datetime(df['timestamp'])
        df.set_index('timestamp', inplace=True)
    elif 'date' in df.columns:
        df['date'] = pd.to_datetime(df['date'])
        df.set_index('date', inplace=True)
    
    # Ensure required columns exist and rename for backtesting library
    required_cols = {'open', 'high', 'low', 'close', 'volume'}
    if not required_cols.issubset(set(df.columns)):
        raise ValueError(f"Missing required columns. Need: {required_cols}")
    
    # Capitalize column names for backtesting library
    df.rename(columns={
        'open': 'Open',
        'high': 'High',
        'low': 'Low',
        'close': 'Close',
        'volume': 'Volume'
    }, inplace=True)
    
    # Remove any NaN values
    df.dropna(inplace=True)
    
    print(f"✅ Data loaded: {len(df)} bars")
    print(f"📅 Date range: {df.index[0]} to {df.index[-1]}")
    print(f"\nFirst few rows:")
    print(df.head())
    
    return df

# Load the data
# data = load_and_prepare_data(DATA_PATH)
print("\n⚠️ Update DATA_PATH variable above with your CSV file location")

## 3. Technical Indicators & Signal Generation

In [ ]:
def calculate_indicators(df: pd.DataFrame) -> pd.DataFrame:
    """
    Calculate all technical indicators for the strategy.
    
    Includes:
    - EMAs (8, 13, 21 periods)
    - ADX and Directional Indicators
    - MFI (Money Flow Index)
    - ATR (Average True Range)
    - Multiple Stochastics (9, 14, 40, 60 periods)
    """
    data = df.copy()
    
    # EMAs
    data['EMA_8'] = ta.EMA(data['Close'], timeperiod=8)
    data['EMA_13'] = ta.EMA(data['Close'], timeperiod=13)
    data['EMA_21'] = ta.EMA(data['Close'], timeperiod=21)
    
    # ADX and Directional Indicators
    data['ADX'] = ta.ADX(data['High'], data['Low'], data['Close'], timeperiod=14)
    data['DI_Plus'] = ta.PLUS_DI(data['High'], data['Low'], data['Close'], timeperiod=14)
    data['DI_Minus'] = ta.MINUS_DI(data['High'], data['Low'], data['Close'], timeperiod=14)
    
    # MFI
    data['MFI'] = ta.MFI(data['High'], data['Low'], data['Close'], data['Volume'], timeperiod=14)
    
    # ATR
    data['ATR'] = ta.ATR(data['High'], data['Low'], data['Close'], timeperiod=14)
    
    # Multiple Stochastics (K and D lines)
    # Stochastic 9
    data['STOCH_K_9'], data['STOCH_D_9'] = ta.STOCH(
        data['High'], data['Low'], data['Close'],
        fastk_period=9, slowk_period=3, slowd_period=3
    )
    
    # Stochastic 14
    data['STOCH_K_14'], data['STOCH_D_14'] = ta.STOCH(
        data['High'], data['Low'], data['Close'],
        fastk_period=14, slowk_period=3, slowd_period=3
    )
    
    # Stochastic 40
    data['STOCH_K_40'], data['STOCH_D_40'] = ta.STOCH(
        data['High'], data['Low'], data['Close'],
        fastk_period=40, slowk_period=3, slowd_period=3
    )
    
    # Stochastic 60
    data['STOCH_K_60'], data['STOCH_D_60'] = ta.STOCH(
        data['High'], data['Low'], data['Close'],
        fastk_period=60, slowk_period=3, slowd_period=3
    )
    
    # Remove initial NaN values
    data.dropna(inplace=True)
    
    return data

print("✅ Indicator calculation function defined")

In [ ]:
def detect_stochastic_breakout(df: pd.DataFrame, 
                               consolidation_bars: int = 3,
                               oversold_level: float = 20.0,
                               overbought_level: float = 80.0) -> pd.DataFrame:
    """
    Detect Multi-Stochastic Breakout signals.
    
    Logic:
    - All 4 D-lines must be condensed (below oversold or above overbought)
    - Must remain condensed for X consecutive bars (consolidation_bars)
    - Breakout occurs when stochastics move out of the condensed zone
    
    Parameters:
    - consolidation_bars: Number of bars stochastics must remain condensed (3-8)
    - oversold_level: Level below which stochastics are considered oversold
    - overbought_level: Level above which stochastics are considered overbought
    """
    data = df.copy()
    
    # Check if all 4 D-lines are below oversold level
    data['All_Oversold'] = (
        (data['STOCH_D_9'] < oversold_level) &
        (data['STOCH_D_14'] < oversold_level) &
        (data['STOCH_D_40'] < oversold_level) &
        (data['STOCH_D_60'] < oversold_level)
    )
    
    # Check if all 4 D-lines are above overbought level
    data['All_Overbought'] = (
        (data['STOCH_D_9'] > overbought_level) &
        (data['STOCH_D_14'] > overbought_level) &
        (data['STOCH_D_40'] > overbought_level) &
        (data['STOCH_D_60'] > overbought_level)
    )
    
    # Count consecutive bars in consolidation
    data['Oversold_Consecutive'] = (
        data['All_Oversold']
        .rolling(window=consolidation_bars, min_periods=consolidation_bars)
        .sum()
    )
    
    data['Overbought_Consecutive'] = (
        data['All_Overbought']
        .rolling(window=consolidation_bars, min_periods=consolidation_bars)
        .sum()
    )
    
    # Detect breakout: Previous bar was consolidating, current bar is not
    data['Stoch_Breakout_Buy'] = (
        (data['Oversold_Consecutive'].shift(1) >= consolidation_bars) &
        (~data['All_Oversold'])
    )
    
    data['Stoch_Breakout_Sell'] = (
        (data['Overbought_Consecutive'].shift(1) >= consolidation_bars) &
        (~data['All_Overbought'])
    )
    
    # Calculate average stochastic value for reference
    data['Stoch_Avg'] = (
        data['STOCH_D_9'] + data['STOCH_D_14'] + 
        data['STOCH_D_40'] + data['STOCH_D_60']
    ) / 4
    
    return data

print("✅ Stochastic breakout detection function defined")

## 4. Consolidated Strategy Class

In [ ]:
class Rip2AlphaStochasticStrategy(Strategy):
    """
    Consolidated Rip2Alpha strategy with integrated Multi-Stochastic Breakout.
    
    Signal Types:
    1. TSBuy/TSSell - Original Trend Shift signals
    2. Continuation signals - Trend continuation entries
    3. Stochastic Breakout - Multi-timeframe stochastic breakout signals
    4. TotalSignal - Combined signal strength
    """
    
    # Strategy Parameters (optimizable)
    signal_sensitivity = 1.0
    adx_threshold = 20.0
    mfi_high_threshold = 70.0
    mfi_low_threshold = 30.0
    stop_loss_atr_multiplier = 2.0
    take_profit_atr_multiplier = 3.0
    stoch_consolidation_bars = 5
    stoch_oversold = 20.0
    stoch_overbought = 80.0
    use_stoch_filter = True
    
    def init(self):
        """Initialize indicators and signals."""
        # Pre-calculate all indicators
        close = self.data.Close
        high = self.data.High
        low = self.data.Low
        
        # Original strategy signals (TSBuy/TSSell)
        self.ts_buy_signal = self.I(self._calculate_ts_buy)
        self.ts_sell_signal = self.I(self._calculate_ts_sell)
        
        # Continuation signals
        self.cont_buy_signal = self.I(self._calculate_cont_buy)
        self.cont_sell_signal = self.I(self._calculate_cont_sell)
        
        # Stochastic breakout signals
        self.stoch_buy_signal = self.I(self._calculate_stoch_buy)
        self.stoch_sell_signal = self.I(self._calculate_stoch_sell)
        
        # Combined total signal
        self.total_buy_signal = self.I(self._calculate_total_buy)
        self.total_sell_signal = self.I(self._calculate_total_sell)
        
        # ATR for stop loss and take profit
        self.atr = self.I(lambda: self.data.df['ATR'])
    
    def _calculate_ts_buy(self):
        """Original Trend Shift Buy signal logic."""
        df = self.data.df
        
        # Bullish trend shift conditions
        ema_bullish = (df['EMA_8'] > df['EMA_13']) & (df['EMA_13'] > df['EMA_21'])
        adx_strong = df['ADX'] > self.adx_threshold
        di_bullish = df['DI_Plus'] > df['DI_Minus']
        mfi_not_overbought = df['MFI'] < self.mfi_high_threshold
        
        # Combine conditions
        signal = (
            ema_bullish & 
            adx_strong & 
            di_bullish & 
            mfi_not_overbought
        )
        
        # Apply sensitivity
        if self.signal_sensitivity != 1.0:
            # Adjust ADX threshold based on sensitivity
            adjusted_adx = df['ADX'] > (self.adx_threshold / self.signal_sensitivity)
            signal = ema_bullish & adjusted_adx & di_bullish & mfi_not_overbought
        
        return signal.astype(int).values
    
    def _calculate_ts_sell(self):
        """Original Trend Shift Sell signal logic."""
        df = self.data.df
        
        # Bearish trend shift conditions
        ema_bearish = (df['EMA_8'] < df['EMA_13']) & (df['EMA_13'] < df['EMA_21'])
        adx_strong = df['ADX'] > self.adx_threshold
        di_bearish = df['DI_Minus'] > df['DI_Plus']
        mfi_not_oversold = df['MFI'] > self.mfi_low_threshold
        
        # Combine conditions
        signal = (
            ema_bearish & 
            adx_strong & 
            di_bearish & 
            mfi_not_oversold
        )
        
        # Apply sensitivity
        if self.signal_sensitivity != 1.0:
            adjusted_adx = df['ADX'] > (self.adx_threshold / self.signal_sensitivity)
            signal = ema_bearish & adjusted_adx & di_bearish & mfi_not_oversold
        
        return signal.astype(int).values
    
    def _calculate_cont_buy(self):
        """Continuation Buy signal logic."""
        df = self.data.df
        
        # Already in uptrend, looking for continuation
        in_uptrend = (df['EMA_8'] > df['EMA_21'])
        pullback = (df['Close'] <= df['EMA_13']) & (df['Close'].shift(1) > df['EMA_13'].shift(1))
        adx_moderate = df['ADX'] > (self.adx_threshold * 0.8)
        
        signal = in_uptrend & pullback & adx_moderate
        
        return signal.astype(int).values
    
    def _calculate_cont_sell(self):
        """Continuation Sell signal logic."""
        df = self.data.df
        
        # Already in downtrend, looking for continuation
        in_downtrend = (df['EMA_8'] < df['EMA_21'])
        pullback = (df['Close'] >= df['EMA_13']) & (df['Close'].shift(1) < df['EMA_13'].shift(1))
        adx_moderate = df['ADX'] > (self.adx_threshold * 0.8)
        
        signal = in_downtrend & pullback & adx_moderate
        
        return signal.astype(int).values
    
    def _calculate_stoch_buy(self):
        """Stochastic Breakout Buy signal logic."""
        df = self.data.df
        
        # Get stochastic breakout buy signal from pre-calculated data
        if 'Stoch_Breakout_Buy' in df.columns:
            return df['Stoch_Breakout_Buy'].astype(int).values
        else:
            return np.zeros(len(df), dtype=int)
    
    def _calculate_stoch_sell(self):
        """Stochastic Breakout Sell signal logic."""
        df = self.data.df
        
        # Get stochastic breakout sell signal from pre-calculated data
        if 'Stoch_Breakout_Sell' in df.columns:
            return df['Stoch_Breakout_Sell'].astype(int).values
        else:
            return np.zeros(len(df), dtype=int)
    
    def _calculate_total_buy(self):
        """Combined Buy signal with weighting."""
        # Weight different signal types
        ts_weight = 2.0  # Trend shift signals are strongest
        stoch_weight = 1.5  # Stochastic breakouts are strong
        cont_weight = 1.0  # Continuation signals are supplementary
        
        total = (
            self.ts_buy_signal * ts_weight +
            self.stoch_buy_signal * stoch_weight +
            self.cont_buy_signal * cont_weight
        )
        
        return total
    
    def _calculate_total_sell(self):
        """Combined Sell signal with weighting."""
        ts_weight = 2.0
        stoch_weight = 1.5
        cont_weight = 1.0
        
        total = (
            self.ts_sell_signal * ts_weight +
            self.stoch_sell_signal * stoch_weight +
            self.cont_sell_signal * cont_weight
        )
        
        return total
    
    def next(self):
        """Execute trading logic on each bar."""
        # Skip if not enough data
        if len(self.data) < 100:
            return
        
        # Get current values
        current_atr = self.atr[-1]
        
        # Entry conditions
        if self.use_stoch_filter:
            # Use combined total signal
            buy_condition = self.total_buy_signal[-1] >= 2.0  # Threshold for entry
            sell_condition = self.total_sell_signal[-1] >= 2.0
        else:
            # Use only original TS signals
            buy_condition = self.ts_buy_signal[-1] == 1
            sell_condition = self.ts_sell_signal[-1] == 1
        
        # Execute trades
        if not self.position:
            if buy_condition:
                # Calculate stop loss and take profit
                sl = self.data.Close[-1] - (current_atr * self.stop_loss_atr_multiplier)
                tp = self.data.Close[-1] + (current_atr * self.take_profit_atr_multiplier)
                
                self.buy(sl=sl, tp=tp)
            
            elif sell_condition:
                sl = self.data.Close[-1] + (current_atr * self.stop_loss_atr_multiplier)
                tp = self.data.Close[-1] - (current_atr * self.take_profit_atr_multiplier)
                
                self.sell(sl=sl, tp=tp)
        
        # Exit conditions (opposite signal)
        elif self.position.is_long and sell_condition:
            self.position.close()
        
        elif self.position.is_short and buy_condition:
            self.position.close()

print("✅ Strategy class defined")

## 5. Backtesting Functions

In [ ]:
def prepare_data_for_backtest(df: pd.DataFrame, 
                              consolidation_bars: int = 5) -> pd.DataFrame:
    """
    Prepare data with all indicators and signals.
    """
    # Calculate basic indicators
    data = calculate_indicators(df)
    
    # Add stochastic breakout signals
    data = detect_stochastic_breakout(
        data,
        consolidation_bars=consolidation_bars
    )
    
    return data

def run_backtest(data: pd.DataFrame, 
                 cash: float = 100000,
                 commission: float = 0.002,
                 **strategy_params) -> Tuple[Any, Backtest]:
    """
    Run backtest with given parameters.
    
    Returns:
        (stats, backtest_object)
    """
    bt = Backtest(
        data,
        Rip2AlphaStochasticStrategy,
        cash=cash,
        commission=commission,
        exclusive_orders=True
    )
    
    stats = bt.run(**strategy_params)
    
    return stats, bt

def print_backtest_results(stats: Any, title: str = "Backtest Results"):
    """
    Print formatted backtest results.
    """
    print(f"\n{'='*60}")
    print(f"{title:^60}")
    print(f"{'='*60}")
    
    metrics = [
        ('Return [%]', stats['Return [%]']),
        ('Sharpe Ratio', stats['Sharpe Ratio']),
        ('Max. Drawdown [%]', stats['Max. Drawdown [%]']),
        ('Win Rate [%]', stats['Win Rate [%]']),
        ('# Trades', stats['# Trades']),
        ('Avg. Trade [%]', stats.get('Avg. Trade [%]', 'N/A')),
        ('Profit Factor', stats.get('Profit Factor', 'N/A')),
    ]
    
    for metric, value in metrics:
        if isinstance(value, (int, float)):
            print(f"{metric:<25} {value:>15.2f}")
        else:
            print(f"{metric:<25} {value:>15}")
    
    print(f"{'='*60}\n")

print("✅ Backtesting functions defined")

## 6. Optimization Framework

In [ ]:
def grid_search_optimization(data: pd.DataFrame,
                            param_grid: Dict[str, List],
                            metric: str = 'Sharpe Ratio',
                            max_iterations: int = 100) -> pd.DataFrame:
    """
    Perform grid search optimization over parameter space.
    
    Parameters:
        data: Prepared data with all indicators
        param_grid: Dictionary of parameters to optimize
        metric: Metric to optimize ('Sharpe Ratio', 'Return [%]', etc.)
        max_iterations: Maximum number of parameter combinations to test
    
    Returns:
        DataFrame with results sorted by metric
    """
    from sklearn.model_selection import ParameterGrid
    
    # Generate all parameter combinations
    param_combinations = list(ParameterGrid(param_grid))
    
    # Limit iterations if needed
    if len(param_combinations) > max_iterations:
        print(f"⚠️ Limiting to {max_iterations} iterations (of {len(param_combinations)} possible)")
        import random
        param_combinations = random.sample(param_combinations, max_iterations)
    else:
        print(f"Testing {len(param_combinations)} parameter combinations...")
    
    results = []
    
    for i, params in enumerate(param_combinations, 1):
        try:
            stats, _ = run_backtest(data, **params)
            
            result = params.copy()
            result['Return [%]'] = stats['Return [%]']
            result['Sharpe Ratio'] = stats['Sharpe Ratio']
            result['Max. Drawdown [%]'] = stats['Max. Drawdown [%]']
            result['Win Rate [%]'] = stats['Win Rate [%]']
            result['# Trades'] = stats['# Trades']
            
            results.append(result)
            
            if i % 10 == 0:
                print(f"Progress: {i}/{len(param_combinations)} ({i/len(param_combinations)*100:.1f}%)")
        
        except Exception as e:
            print(f"Error with params {params}: {e}")
            continue
    
    # Convert to DataFrame and sort
    results_df = pd.DataFrame(results)
    
    # Sort by metric (descending for most metrics)
    if 'Drawdown' in metric:
        results_df = results_df.sort_values(metric, ascending=True)
    else:
        results_df = results_df.sort_values(metric, ascending=False)
    
    print(f"\n✅ Optimization complete!")
    
    return results_df

print("✅ Grid search optimization function defined")

In [ ]:
def bayesian_optimization(data: pd.DataFrame,
                         search_space: Dict,
                         n_calls: int = 50,
                         metric: str = 'Sharpe Ratio') -> Dict:
    """
    Perform Bayesian optimization using scikit-optimize.
    
    More efficient than grid search for large parameter spaces.
    """
    if not SKOPT_AVAILABLE:
        print("❌ Bayesian optimization not available. Install scikit-optimize.")
        return None
    
    from skopt import gp_minimize
    from skopt.space import Real, Integer
    from skopt.utils import use_named_args
    
    # Define the objective function
    @use_named_args(list(search_space.values()))
    def objective(**params):
        try:
            stats, _ = run_backtest(data, **params)
            
            # Return negative for minimization (we want to maximize)
            if 'Drawdown' in metric:
                return stats[metric]  # Minimize drawdown
            else:
                return -stats[metric]  # Maximize other metrics
        
        except Exception as e:
            print(f"Error in optimization: {e}")
            return 1e10  # Return large penalty
    
    print(f"Starting Bayesian optimization with {n_calls} iterations...")
    
    # Run optimization
    result = gp_minimize(
        objective,
        list(search_space.values()),
        n_calls=n_calls,
        random_state=42,
        verbose=True
    )
    
    # Extract best parameters
    best_params = {}
    for i, (name, _) in enumerate(search_space.items()):
        best_params[name] = result.x[i]
    
    print(f"\n✅ Bayesian optimization complete!")
    print(f"Best {metric}: {-result.fun:.2f}")
    print(f"Best parameters: {best_params}")
    
    return best_params

print("✅ Bayesian optimization function defined")

## 7. Machine Learning Optimization

In [ ]:
def prepare_ml_features(data: pd.DataFrame) -> pd.DataFrame:
    """
    Prepare features for machine learning signal filtering.
    
    Creates additional features from indicators for ML model.
    """
    df = data.copy()
    
    # Price-based features
    df['Returns'] = df['Close'].pct_change()
    df['HL_Ratio'] = (df['High'] - df['Low']) / df['Close']
    df['OC_Ratio'] = (df['Close'] - df['Open']) / df['Close']
    
    # EMA relationships
    df['EMA_8_13_Diff'] = (df['EMA_8'] - df['EMA_13']) / df['Close']
    df['EMA_13_21_Diff'] = (df['EMA_13'] - df['EMA_21']) / df['Close']
    
    # Momentum features
    df['ADX_Change'] = df['ADX'].diff()
    df['MFI_Change'] = df['MFI'].diff()
    
    # Stochastic features
    df['Stoch_Divergence'] = df['STOCH_D_9'] - df['STOCH_D_60']
    df['Stoch_Momentum'] = df['STOCH_D_14'].diff()
    
    # Volume features
    df['Volume_SMA_20'] = df['Volume'].rolling(20).mean()
    df['Volume_Ratio'] = df['Volume'] / df['Volume_SMA_20']
    
    # Create target variable (future returns)
    df['Future_Return_5'] = df['Close'].shift(-5) / df['Close'] - 1
    df['Future_Return_10'] = df['Close'].shift(-10) / df['Close'] - 1
    
    # Binary target: 1 if profitable, 0 if not
    df['Target_5'] = (df['Future_Return_5'] > 0).astype(int)
    df['Target_10'] = (df['Future_Return_10'] > 0).astype(int)
    
    return df

def train_ml_filter(data: pd.DataFrame, target_col: str = 'Target_5') -> RandomForestClassifier:
    """
    Train a machine learning model to filter signals.
    
    Returns a trained model that can predict signal quality.
    """
    from sklearn.model_selection import train_test_split
    from sklearn.ensemble import RandomForestClassifier
    
    df = data.copy()
    
    # Select features
    feature_cols = [
        'ADX', 'MFI', 'DI_Plus', 'DI_Minus',
        'EMA_8_13_Diff', 'EMA_13_21_Diff',
        'Stoch_Avg', 'Stoch_Divergence', 'Stoch_Momentum',
        'HL_Ratio', 'OC_Ratio', 'Volume_Ratio',
        'ADX_Change', 'MFI_Change'
    ]
    
    # Remove NaN values
    df = df.dropna(subset=feature_cols + [target_col])
    
    X = df[feature_cols]
    y = df[target_col]
    
    # Split data
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=42, shuffle=False  # No shuffle to maintain time order
    )
    
    # Train model
    print("Training Random Forest classifier...")
    model = RandomForestClassifier(
        n_estimators=100,
        max_depth=10,
        min_samples_split=50,
        random_state=42,
        n_jobs=-1
    )
    
    model.fit(X_train, y_train)
    
    # Evaluate
    train_score = model.score(X_train, y_train)
    test_score = model.score(X_test, y_test)
    
    print(f"\nModel Performance:")
    print(f"Training Accuracy: {train_score:.3f}")
    print(f"Testing Accuracy: {test_score:.3f}")
    
    # Feature importance
    feature_importance = pd.DataFrame({
        'Feature': feature_cols,
        'Importance': model.feature_importances_
    }).sort_values('Importance', ascending=False)
    
    print("\nTop 5 Most Important Features:")
    print(feature_importance.head())
    
    return model

print("✅ Machine learning functions defined")

## 8. Analysis and Visualization

In [ ]:
def analyze_signals(data: pd.DataFrame):
    """
    Analyze and visualize strategy signals.
    """
    # Count signals
    ts_buy_count = data['Stoch_Breakout_Buy'].sum()
    ts_sell_count = data['Stoch_Breakout_Sell'].sum()
    
    print(f"\n📊 Signal Analysis:")
    print(f"{'='*50}")
    print(f"Stochastic Breakout Buy Signals: {ts_buy_count}")
    print(f"Stochastic Breakout Sell Signals: {ts_sell_count}")
    print(f"Total Signals: {ts_buy_count + ts_sell_count}")
    print(f"{'='*50}\n")
    
    # Plot signals on price chart
    fig, axes = plt.subplots(3, 1, figsize=(16, 12), sharex=True)
    
    # Price and EMAs
    axes[0].plot(data.index, data['Close'], label='Close', linewidth=1.5, color='black')
    axes[0].plot(data.index, data['EMA_8'], label='EMA 8', alpha=0.7)
    axes[0].plot(data.index, data['EMA_13'], label='EMA 13', alpha=0.7)
    axes[0].plot(data.index, data['EMA_21'], label='EMA 21', alpha=0.7)
    
    # Mark signals
    buy_signals = data[data['Stoch_Breakout_Buy']]
    sell_signals = data[data['Stoch_Breakout_Sell']]
    
    axes[0].scatter(buy_signals.index, buy_signals['Close'], 
                   marker='^', s=100, color='green', label='Stoch Buy', zorder=5)
    axes[0].scatter(sell_signals.index, sell_signals['Close'], 
                   marker='v', s=100, color='red', label='Stoch Sell', zorder=5)
    
    axes[0].set_ylabel('Price')
    axes[0].legend(loc='best')
    axes[0].set_title('Price Chart with Stochastic Breakout Signals')
    axes[0].grid(True, alpha=0.3)
    
    # Stochastic oscillators
    axes[1].plot(data.index, data['STOCH_D_9'], label='Stoch D 9', alpha=0.7)
    axes[1].plot(data.index, data['STOCH_D_14'], label='Stoch D 14', alpha=0.7)
    axes[1].plot(data.index, data['STOCH_D_40'], label='Stoch D 40', alpha=0.7)
    axes[1].plot(data.index, data['STOCH_D_60'], label='Stoch D 60', alpha=0.7)
    axes[1].axhline(y=20, color='green', linestyle='--', alpha=0.5, label='Oversold')
    axes[1].axhline(y=80, color='red', linestyle='--', alpha=0.5, label='Overbought')
    axes[1].set_ylabel('Stochastic')
    axes[1].legend(loc='best')
    axes[1].set_title('Multi-Timeframe Stochastic Oscillators')
    axes[1].grid(True, alpha=0.3)
    
    # ADX and MFI
    ax3a = axes[2]
    ax3a.plot(data.index, data['ADX'], label='ADX', color='blue')
    ax3a.axhline(y=20, color='blue', linestyle='--', alpha=0.5)
    ax3a.set_ylabel('ADX', color='blue')
    ax3a.legend(loc='upper left')
    ax3a.grid(True, alpha=0.3)
    
    ax3b = ax3a.twinx()
    ax3b.plot(data.index, data['MFI'], label='MFI', color='orange')
    ax3b.axhline(y=30, color='green', linestyle='--', alpha=0.5)
    ax3b.axhline(y=70, color='red', linestyle='--', alpha=0.5)
    ax3b.set_ylabel('MFI', color='orange')
    ax3b.legend(loc='upper right')
    
    axes[2].set_title('ADX and MFI')
    axes[2].set_xlabel('Date')
    
    plt.tight_layout()
    plt.show()

def compare_strategies(results: Dict[str, Any]):
    """
    Compare multiple strategy configurations.
    
    Parameters:
        results: Dictionary with strategy names as keys and stats as values
    """
    comparison_data = []
    
    for name, stats in results.items():
        comparison_data.append({
            'Strategy': name,
            'Return [%]': stats['Return [%]'],
            'Sharpe Ratio': stats['Sharpe Ratio'],
            'Max DD [%]': stats['Max. Drawdown [%]'],
            'Win Rate [%]': stats['Win Rate [%]'],
            '# Trades': stats['# Trades']
        })
    
    comparison_df = pd.DataFrame(comparison_data)
    
    print("\n" + "="*80)
    print("STRATEGY COMPARISON")
    print("="*80)
    print(comparison_df.to_string(index=False))
    print("="*80 + "\n")
    
    # Visualize comparison
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Return
    axes[0, 0].bar(comparison_df['Strategy'], comparison_df['Return [%]'])
    axes[0, 0].set_title('Return [%]')
    axes[0, 0].tick_params(axis='x', rotation=45)
    axes[0, 0].grid(True, alpha=0.3)
    
    # Sharpe Ratio
    axes[0, 1].bar(comparison_df['Strategy'], comparison_df['Sharpe Ratio'])
    axes[0, 1].set_title('Sharpe Ratio')
    axes[0, 1].tick_params(axis='x', rotation=45)
    axes[0, 1].grid(True, alpha=0.3)
    
    # Max Drawdown
    axes[1, 0].bar(comparison_df['Strategy'], comparison_df['Max DD [%]'])
    axes[1, 0].set_title('Max Drawdown [%]')
    axes[1, 0].tick_params(axis='x', rotation=45)
    axes[1, 0].grid(True, alpha=0.3)
    
    # Number of Trades
    axes[1, 1].bar(comparison_df['Strategy'], comparison_df['# Trades'])
    axes[1, 1].set_title('Number of Trades')
    axes[1, 1].tick_params(axis='x', rotation=45)
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

print("✅ Analysis and visualization functions defined")

## 9. Complete Workflow Execution

### 9.1 Load and Prepare Data

In [ ]:
# UPDATE THIS PATH!
DATA_PATH = '/content/drive/MyDrive/trading_data/your_data.csv'

# Load data
raw_data = load_and_prepare_data(DATA_PATH)

# Prepare with all indicators and signals
data = prepare_data_for_backtest(
    raw_data,
    consolidation_bars=5  # Test different values: 3-8
)

print(f"\n✅ Data prepared with {len(data)} bars")
print(f"Columns: {list(data.columns)}")

### 9.2 Analyze Signals

In [ ]:
# Analyze and visualize signals
analyze_signals(data)

### 9.3 Run Baseline Backtest

In [ ]:
# Run baseline backtest with default parameters
print("Running baseline backtest...\n")

baseline_stats, baseline_bt = run_backtest(
    data,
    signal_sensitivity=1.0,
    adx_threshold=20.0,
    mfi_high_threshold=70.0,
    mfi_low_threshold=30.0,
    stop_loss_atr_multiplier=2.0,
    take_profit_atr_multiplier=3.0,
    use_stoch_filter=True
)

print_backtest_results(baseline_stats, "Baseline Strategy (with Stochastics)")

# Plot results
baseline_bt.plot()

### 9.4 Compare With and Without Stochastic Filter

In [ ]:
# Test without stochastic filter (original strategy only)
print("Running original strategy (no stochastic filter)...\n")

original_stats, original_bt = run_backtest(
    data,
    use_stoch_filter=False
)

print_backtest_results(original_stats, "Original Strategy (TS signals only)")

# Compare strategies
compare_strategies({
    'Original (TS only)': original_stats,
    'With Stochastics': baseline_stats
})

### 9.5 Parameter Optimization - Grid Search

In [ ]:
# Define parameter grid for optimization
param_grid = {
    'signal_sensitivity': [0.8, 1.0, 1.2, 1.5],
    'adx_threshold': [15.0, 20.0, 25.0],
    'mfi_high_threshold': [65.0, 70.0, 75.0],
    'mfi_low_threshold': [25.0, 30.0, 35.0],
    'stop_loss_atr_multiplier': [1.5, 2.0, 2.5],
    'take_profit_atr_multiplier': [2.5, 3.0, 4.0, 5.0],
    'stoch_consolidation_bars': [3, 5, 7],
    'use_stoch_filter': [True]
}

print("Starting grid search optimization...\n")

# Run optimization
optimization_results = grid_search_optimization(
    data,
    param_grid,
    metric='Sharpe Ratio',
    max_iterations=100  # Increase for more thorough search
)

# Display top 10 results
print("\n📊 TOP 10 OPTIMIZATION RESULTS:")
print("="*100)
print(optimization_results.head(10).to_string(index=False))
print("="*100)

### 9.6 Test Best Parameters

In [ ]:
# Extract best parameters
best_params = optimization_results.iloc[0].to_dict()

# Remove metric columns
metric_cols = ['Return [%]', 'Sharpe Ratio', 'Max. Drawdown [%]', 'Win Rate [%]', '# Trades']
best_params = {k: v for k, v in best_params.items() if k not in metric_cols}

print("\n🏆 BEST OPTIMIZED PARAMETERS:")
print("="*50)
for param, value in best_params.items():
    print(f"{param:<30} {value}")
print("="*50)

# Run backtest with best parameters
print("\nTesting optimized parameters...\n")

optimized_stats, optimized_bt = run_backtest(data, **best_params)

print_backtest_results(optimized_stats, "Optimized Strategy")

# Compare all strategies
compare_strategies({
    'Original (TS only)': original_stats,
    'Baseline (with Stoch)': baseline_stats,
    'Optimized': optimized_stats
})

# Plot optimized results
optimized_bt.plot()

### 9.7 Machine Learning Enhancement (Optional)

In [ ]:
# Prepare ML features
print("Preparing machine learning features...\n")

ml_data = prepare_ml_features(data)

# Train ML model to filter signals
ml_model = train_ml_filter(ml_data, target_col='Target_5')

print("\n✅ ML model trained successfully!")
print("\nThis model can be used to further filter signals based on learned patterns.")

### 9.8 Bayesian Optimization (Advanced)

In [ ]:
# Only run if scikit-optimize is available
if SKOPT_AVAILABLE:
    from skopt.space import Real, Integer
    
    # Define search space
    search_space = {
        'signal_sensitivity': Real(0.5, 2.0, name='signal_sensitivity'),
        'adx_threshold': Real(10.0, 30.0, name='adx_threshold'),
        'mfi_high_threshold': Real(60.0, 80.0, name='mfi_high_threshold'),
        'mfi_low_threshold': Real(20.0, 40.0, name='mfi_low_threshold'),
        'stop_loss_atr_multiplier': Real(1.0, 3.0, name='stop_loss_atr_multiplier'),
        'take_profit_atr_multiplier': Real(2.0, 6.0, name='take_profit_atr_multiplier'),
        'stoch_consolidation_bars': Integer(3, 8, name='stoch_consolidation_bars'),
    }
    
    # Run Bayesian optimization
    bayesian_best_params = bayesian_optimization(
        data,
        search_space,
        n_calls=50,
        metric='Sharpe Ratio'
    )
    
    # Test Bayesian optimized parameters
    if bayesian_best_params:
        bayesian_best_params['use_stoch_filter'] = True
        
        bayesian_stats, bayesian_bt = run_backtest(data, **bayesian_best_params)
        
        print_backtest_results(bayesian_stats, "Bayesian Optimized Strategy")
        
        # Final comparison
        compare_strategies({
            'Original (TS only)': original_stats,
            'Baseline (with Stoch)': baseline_stats,
            'Grid Search Optimized': optimized_stats,
            'Bayesian Optimized': bayesian_stats
        })
else:
    print("⚠️ Bayesian optimization not available. Install scikit-optimize to use this feature.")

### 9.9 Save Results

In [ ]:
from google.colab import files
import json

# Generate timestamp
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

# Save optimization results
optimization_results.head(20).to_csv(f'optimization_results_{timestamp}.csv', index=False)
print(f"✅ Optimization results saved to: optimization_results_{timestamp}.csv")

# Save best parameters
with open(f'best_parameters_{timestamp}.json', 'w') as f:
    json.dump(best_params, f, indent=4)
print(f"✅ Best parameters saved to: best_parameters_{timestamp}.json")

# Save performance summary
performance_summary = {
    'baseline': dict(baseline_stats),
    'optimized': dict(optimized_stats),
    'original_no_stoch': dict(original_stats)
}

with open(f'performance_summary_{timestamp}.json', 'w') as f:
    json.dump(performance_summary, f, indent=4)
print(f"✅ Performance summary saved to: performance_summary_{timestamp}.json")

# Download files
files.download(f'optimization_results_{timestamp}.csv')
files.download(f'best_parameters_{timestamp}.json')
files.download(f'performance_summary_{timestamp}.json')

print("\n✅ All results saved and downloaded!")

## 10. Summary and Next Steps

This consolidated notebook provides:

1. ✅ **Original Strategy**: TSBuy/TSSell signals based on EMA, ADX, DI, and MFI
2. ✅ **Continuation Signals**: Pullback entries in established trends
3. ✅ **Multi-Stochastic Breakout**: 4 D-line stochastics (9, 14, 40, 60) with consolidation detection
4. ✅ **Combined Signals**: TotalSignal that weights all signal types
5. ✅ **Comprehensive Backtesting**: Full performance analysis
6. ✅ **Grid Search Optimization**: Tests multiple parameter combinations
7. ✅ **Bayesian Optimization**: Advanced optimization for large parameter spaces
8. ✅ **Machine Learning**: Signal filtering based on learned patterns
9. ✅ **Performance Comparison**: Side-by-side strategy comparison

### Recommended Next Steps:

1. **Test Different Timeframes**: Try 1-min, 5-min, 15-min data
2. **Adjust Consolidation Bars**: Test stoch_consolidation_bars from 3 to 8
3. **Optimize Take Profit**: Focus on take_profit_atr_multiplier optimization
4. **Walk-Forward Analysis**: Split data into training and validation periods
5. **Live Paper Trading**: Test optimized parameters in real-time
6. **Add More Filters**: Consider volume, volatility, or time-of-day filters

### Key Parameters to Focus On:

- **take_profit_atr_multiplier**: Most impactful for profitability
- **stoch_consolidation_bars**: Controls breakout signal timing (3-8 bars)
- **signal_sensitivity**: Adjusts how aggressive/conservative the strategy is
- **use_stoch_filter**: Toggle stochastic breakout logic on/off
